# PROTEUS: как устроены данные Track 2

Три мысленные команды по 4-секундным окнам ЭЭГ: **представление движения**, **устный счёт**, **придумывание слов**.
Окна подготовлены официальным пайплайном (`python -m bci.data --data-dir data`): 41 канал, 120 Гц, микровольты.
Участники разбиты на train / val / test, ни один человек не попадает в две части.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import welch
from bci.data import load_split, SFREQ

DATA_DIR = "../data"
train, val, test = (load_split(s, DATA_DIR) for s in ["train", "val", "test"])
for name, w in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} X={w.X.shape}  subjects={len(set(w.subject))}  sessions={len(set(w.stream))}")

Каждая сессия участника — это **поток** (stream). В сессии есть прогоны с двумя интерфейсами (Graz и BrainHero), это **контекст**. Метрика усредняется по ячейкам *участник × сессия × интерфейс*.

In [ ]:
pd.crosstab([train.subject, train.session], train.context)

In [ ]:
pd.crosstab(train.subject, train.y).rename(columns={0: "class 0", 1: "class 1", 2: "class 2"})

Одно окно и средний спектр по классам: различия между классами есть, но они небольшие.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
t = np.arange(train.X.shape[-1]) / SFREQ
for k, ch in enumerate(range(0, 41, 8)):
    a.plot(t, train.X[0, ch] + 40 * k, lw=0.7)
a.set(yticks=40 * np.arange(6), yticklabels=train.ch_names[::8], xlabel="s", title=f"окно 0, класс {train.y[0]}")

f, psd = welch(train.X, fs=SFREQ, nperseg=240)
for c in range(3):
    b.semilogy(f, psd[train.y == c].mean((0, 1)), label=f"class {c}")
b.set(xlim=(1, 45), xlabel="Hz", title="средний спектр"); b.legend()
plt.tight_layout()

Главная проблема — **сдвиг между людьми**. На карте ковариаций окна группируются по участникам, а не по классам.
Если «перецентрировать» каждую сессию относительно её собственной средней ковариации, облака разных людей совмещаются.

In [ ]:
from pyriemann.estimation import Covariances
from bci.riemann import mean_riemann, invsqrtm
from pyriemann.tangentspace import TangentSpace
from sklearn.decomposition import PCA

covs = Covariances("oas").transform(train.X.astype(np.float64))
recentered = covs.copy()
for idx in train.streams():
    W = invsqrtm(mean_riemann(covs[idx]))
    recentered[idx] = W @ covs[idx] @ W

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
subj_id = pd.factorize(train.subject)[0]
for ax, C, title in [(axes[0], covs, "как есть"), (axes[1], recentered, "после перецентрирования сессий")]:
    z = PCA(2).fit_transform(TangentSpace().fit_transform(C))
    ax.scatter(*z.T, c=subj_id, cmap="tab10", s=4)
    ax.set_title(f"{title} (цвет = участник)")
plt.tight_layout()